In [10]:
import numpy as np
from PIL import Image
import os
from random import shuffle
from typing import Tuple, List, Optional

class MLPErrorCorrection:
    def __init__(self, input_size: int = 3, output_size: int = 3, learning_rate: float = 0.01) -> None:
        """Инициализация многослойного перцептрона"""
        self.weights: np.ndarray = np.random.randn(input_size, output_size) * 0.01
        self.learning_rate: float = learning_rate
        self.mean: Optional[np.ndarray] = None
        self.std: Optional[np.ndarray] = None
        
    def softmax(self, x: np.ndarray) -> np.ndarray:
        """Функция активации softmax"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def forward(self, X: np.ndarray) -> np.ndarray:
        """Прямой проход через сеть"""
        return self.softmax(np.dot(X, self.weights)) 
    
    def normalize(self, X: np.ndarray) -> np.ndarray:
        """Нормализация входных данных"""
        if self.mean is None or self.std is None:
            raise ValueError("Сначала вызовите fit()")
        return (X - self.mean) / self.std
    
    def fit(self, X_train: np.ndarray, y_train: np.ndarray, epochs: int = 100, 
            min_error: float = 0.01, min_weight_change: float = 1e-5) -> None:
        """Обучение модели"""
        # Вычисляем параметры нормализации
        self.mean = X_train.mean(axis=0)
        self.std = X_train.std(axis=0) + 1e-6
        
        # Нормализуем данные
        X_train_norm = (X_train - self.mean) / self.std
        
        prev_weights = self.weights.copy()
        for epoch in range(epochs):
            # Прямой проход
            output = self.forward(X_train_norm)
            
            # Вычисление ошибки (y_train - output)
            error = y_train - output
            
            # Обновление весов (коррекция по ошибке)
            delta = self.learning_rate * X_train_norm.T.dot(error)
            self.weights += delta
            
            # Проверка условий остановки
            total_error = np.sum(np.abs(error))
            weight_change = np.max(np.abs(self.weights - prev_weights))
            
            if total_error < min_error:
                print(f"Остановка на эпохе {epoch}: ошибка < {min_error}")
                break
                
            if weight_change < min_weight_change:
                print(f"Остановка на эпохе {epoch}: изменение весов < {min_weight_change}")
                break
                
            prev_weights = self.weights.copy()
            
            if epoch % (epochs // 10) == 0:
                print(f"Эпоха {epoch}, Ошибка: {total_error:.4f}, Изменение весов: {weight_change:.6f}")
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Предсказание классов для входных данных"""
        X_norm = (X - self.mean) / self.std
        output = self.forward(X_norm)
        return np.argmax(output, axis=1)  # Возвращаем индекс класса с максимальной вероятностью
    
    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray) -> float:
        """Оценка точности модели на тестовых данных"""
        predictions = self.predict(X_test)
        true_labels = np.argmax(y_test, axis=1)
        accuracy = np.mean(predictions == true_labels)
        return accuracy

def extract_features(image_path: str) -> np.ndarray:
    """Извлечение признаков изображения"""
    try:
        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img)
        
        brightness = np.mean(img_array) / 255.0
        green_ratio = np.mean(img_array[:, :, 1]) / 255.0
        gray_img = img.convert('L')
        contrast = np.std(np.array(gray_img)) / 255.0
        
        return np.array([brightness, green_ratio, contrast])
    except Exception as e:
        print(f"Ошибка обработки {image_path}: {str(e)}")
        return np.zeros(3)

def load_dataset(data_dir: str, test_size: float = 0.2) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Загрузка и разделение датасета на обучающую и тестовую выборки"""
    X: List[np.ndarray] = []
    y: List[np.ndarray] = []
    class_names = ['forest', 'desert'] 
    
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        for filename in os.listdir(class_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_dir, filename)
                features = extract_features(img_path)
                X.append(features)
                # One-hot encoding для меток
                one_hot = np.zeros(len(class_names))
                one_hot[label] = 1
                y.append(one_hot)
    
    combined = list(zip(X, y))
    shuffle(combined)
    X, y = zip(*combined)
    
    split_idx = int(len(X) * (1 - test_size))
    X_train = np.array(X[:split_idx])
    y_train = np.array(y[:split_idx])
    X_test = np.array(X[split_idx:])
    y_test = np.array(y[split_idx:])
    
    return X_train, y_train, X_test, y_test

In [11]:
# Конфигурация
data_dir = "origins/data"

# Разделение на train/test
X_train, y_train, X_test, y_test = load_dataset(data_dir)

# Создание и обучение перцептрона
mlp = MLPErrorCorrection(input_size=3, output_size=2, learning_rate=0.1)
mlp.fit(X_train, y_train, epochs=10000)

# Оценка точности
train_accuracy = mlp.evaluate(X_train, y_train)
test_accuracy = mlp.evaluate(X_test, y_test)
print(f"\nТочность на обучающей выборке: {train_accuracy*100:.2f}%")
print(f"Точность на тестовой выборке: {test_accuracy*100:.2f}%")

Эпоха 0, Ошибка: 643.3080, Изменение весов: 22.832260
Эпоха 1000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 2000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 3000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 4000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 5000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 6000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 7000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 8000, Ошибка: 126.3479, Изменение весов: 5.499778
Эпоха 9000, Ошибка: 126.3479, Изменение весов: 5.499778

Точность на обучающей выборке: 90.02%
Точность на тестовой выборке: 96.27%
